# Cell 1: Imports and Configurations

In [ ]:
import os
import math
import torch
import shutil
import spacy
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM


# --- SPEED OPTIMIZATIONS ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


TRAIN_CSV = "train_dataset.csv"
TEST_CSV = "test_dataset.csv"


TRAIN_TASKS_CSV = "train_tasks_expanded.csv"
LOCAL_TRAIN_VECS = "train_vectors_unified.pt"

TEST_TASKS_CSV = "test_tasks_expanded.csv"
LOCAL_TEST_VECS = "test_vectors_unified.pt"


# --- 3. LOAD SPACY ---
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Cell 2: Load Model

In [ ]:
# --- LOAD LLAMA ---
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", use_fast=True)
tokenizer.padding_side = "right"

# Assign the EOS token to act as the Padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct", device_map="auto", torch_dtype=torch.bfloat16, attn_implementation="sdpa"
)
model.eval()
terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]

# Cell 3: Extraction

In [ ]:
# ==========================================
# STANDALONE VECTOR EXTRACTION
# ==========================================

# --- HELPERS ---
def get_log2_chunks(seq_len):
    if seq_len <= 1: return [(0, seq_len)]
    num_chunks = max(1, int(math.log2(seq_len)))
    chunk_size = seq_len / num_chunks
    chunks = []
    for i in range(num_chunks):
        start = int(i * chunk_size)
        end = int((i + 1) * chunk_size) if i < (num_chunks - 1) else seq_len
        chunks.append((start, end))
    return chunks

def get_entity_weights(text, token_offsets, seq_len):
    doc = nlp(text)
    weights = torch.full((seq_len,), 0.1, device=model.device, dtype=torch.bfloat16)
    entity_spans = [(ent.start_char, ent.end_char) for ent in doc.ents]
    noun_spans = [(chunk.start_char, chunk.end_char) for chunk in doc.noun_chunks]
    all_spans = entity_spans + noun_spans

    for idx, (tok_start, tok_end) in enumerate(token_offsets):
        if tok_start == tok_end: continue
        for (span_start, span_end) in all_spans:
            if tok_start < span_end and tok_end > span_start:
                weights[idx] = 1.0
                break
    return weights

# --- EXTRACTION PIPELINE ---
def extract_rescue_vectors(csv_path, local_save_path, batch_size=32):
    print(f"\nLoading {os.path.basename(csv_path)}...")
    df = pd.read_csv(csv_path)

    original_len = len(df)
    df = df.drop_duplicates(subset=['question_id']).reset_index(drop=True)
    print(f"Dropped {original_len - len(df)} duplicate task rows. Unique contexts to process: {len(df)}")

    df['approx_len'] = df['context'].astype(str).str.len()
    df.sort_values('approx_len', inplace=True)
    df.drop(columns=['approx_len'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    vectors_dict = {}
    target_layers = [8, 16, 24]

    for start_idx in tqdm(range(0, len(df), batch_size), desc="Extracting Batches"):
        batch_df = df.iloc[start_idx:start_idx+batch_size]
        contexts = batch_df['context'].astype(str).tolist()

        ext_inputs = tokenizer(contexts, return_tensors="pt", padding=True, return_offsets_mapping=True).to(model.device)

        with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            ext_outputs = model(
                input_ids=ext_inputs.input_ids,
                attention_mask=ext_inputs.attention_mask,
                output_hidden_states=True
            )

        for i, row in enumerate(batch_df.itertuples()):
            c_id = row.question_id
            seq_len = ext_inputs.attention_mask[i].sum().item()

            offsets = ext_inputs.offset_mapping[i][:seq_len].tolist()
            weights = get_entity_weights(row.context, offsets, seq_len)

            chunk_boundaries = get_log2_chunks(seq_len)
            num_chunks = len(chunk_boundaries)

            chunk_tensor = torch.zeros((num_chunks, len(target_layers), 3, 4096), dtype=torch.bfloat16, device='cpu')
            global_last_tensor = torch.zeros((len(target_layers), 4096), dtype=torch.bfloat16, device='cpu')

            for l_idx, layer_num in enumerate(target_layers):
                actual_hs = ext_outputs.hidden_states[layer_num][i, :seq_len, :]
                global_last_tensor[l_idx, :] = actual_hs[-1, :].cpu()

                for c_idx, (start, end) in enumerate(chunk_boundaries):
                    chunk_weights = weights[start:end].unsqueeze(-1)
                    w_sum = chunk_weights.sum().clamp(min=1e-6)

                    hs_chunk = actual_hs[start:end, :]

                    mu = (hs_chunk * chunk_weights).sum(dim=0) / w_sum
                    var = (chunk_weights * (hs_chunk - mu)**2).sum(dim=0) / w_sum
                    max_vec = (hs_chunk * chunk_weights).max(dim=0).values

                    chunk_tensor[c_idx, l_idx, 0, :] = mu.cpu()
                    chunk_tensor[c_idx, l_idx, 1, :] = var.cpu()
                    chunk_tensor[c_idx, l_idx, 2, :] = max_vec.cpu()

            vectors_dict[c_id] = {
                "chunk_signals": chunk_tensor,
                "global_last": global_last_tensor
            }

        del ext_inputs, ext_outputs
        torch.cuda.empty_cache()

    torch.save(vectors_dict, local_save_path)
    print("✅ Local extraction complete.")

In [ ]:
extract_rescue_vectors(TEST_TASKS_CSV, LOCAL_TEST_VECS, batch_size=32)

In [ ]:
extract_rescue_vectors(TRAIN_TASKS_CSV, LOCAL_TRAIN_VECS, batch_size=32)